In [1]:
import os
import pandas as pd
import dask.dataframe as dd
from tqdm.notebook import tqdm

from utilities import walk_through_files, mem_usage

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_rows', 500)

from dask.distributed import LocalCluster, Client
cluster = LocalCluster(n_workers=4)
client = Client(cluster)  # Connect to distributed cluster and override default
client

Client Scheduler: tcp://127.0.0.1:63469 Dashboard: http://127.0.0.1:63468/status,Cluster Workers: 4 Cores: 12 Memory: 34.29 GB


In [2]:
HOMECOUNT_PATH = os.path.join('intermediate-files', 'device-area-maxcount.csv.gz')
ALTERNATE_DUMP_PATH = os.path.join('..', '..', '..', 'wgw-lf', 'datadump')
SAVEPATH = os.path.join('intermediate-files', 'daily-movecount.csv.gz')

### Get daily movement (long format)

In [3]:
# read in device home
home_ddf = (dd.read_csv(HOMECOUNT_PATH, compression='gzip')
            .rename(columns={'area_id': 'home'})
            .query('count > 30')
           )

home_ddf

,device_xid,home,count
npartitions=1,,,
,int64,int64,int64
,...,...,...


In [4]:
file_lists = tuple(walk_through_files(ALTERNATE_DUMP_PATH, 'csv.gz', fullpath=True))

# Read in records
daily_ddf = dd.read_csv(file_lists, compression='gzip', usecols=['device_xid', 'date', 'area_id'])

daily_ddf

,device_xid,date,area_id
npartitions=27210,,,
,int64,int64,int64
,...,...,...
...,...,...,...
,...,...,...
,...,...,...


In [5]:
# merge
gravity = (daily_ddf.merge(home_ddf, how='left', on='device_xid')
           .groupby(['date', 'home', 'area_id']).size().reset_index()
           .rename(columns={0: 'inflow_count'})
           .astype({'home': 'int'})
          )

gravity

,date,home,area_id,inflow_count
npartitions=1,,,,
,int64,int32,int64,int64
,...,...,...,...


In [6]:
gravity_df = gravity.compute()

In [7]:
gravity_df

,date,home,area_id,inflow_count
0,1577808000,0,0,7
1,1577808000,37,0,3
2,1577808000,105,0,1
3,1577808000,146,0,1
4,1577808000,214,0,11
...,...,...,...,...
2103962,1585584000,283,300,3
2103963,1585584000,296,300,1
2103964,1585584000,298,300,3
2103965,1585584000,299,300,3


In [8]:
mem_usage(gravity_df)

'56.2 (56.2) MB'

In [9]:
gravity_df.to_csv(SAVEPATH, index=False, compression='gzip')

In [10]:
# How many devices included
df = (home_ddf.compute()
      .sort_values('count', ascending=False)
      .drop_duplicates('device_xid')
      .query('count > 30')
     )
len(df)

125589

### Test output with dummy files

In [11]:
dummy_datadump_ddf = pd.read_csv(os.path.join('dummyfiles', 'datadump.csv'))
dummy_home_ddf = pd.read_csv(os.path.join('dummyfiles', 'dummy-home.csv'))

merged_only = dummy_datadump_ddf.merge(dummy_home_ddf, how='left', on='device_xid')

dummy_output = (dummy_datadump_ddf.merge(dummy_home_ddf, how='left', on='device_xid')
                .groupby(['date', 'home', 'area_id']).size().reset_index()
                .rename(columns={0: 'inflow_count'})
                .astype({'home': 'int'})
               )

dummy_output

,date,home,area_id,inflow_count
0,1,1,1,2
1,1,1,2,1
2,1,2,1,1
3,1,2,2,1
4,2,1,1,2
5,2,1,2,1
6,2,2,1,1
7,2,2,2,1
8,3,1,1,1
9,3,1,3,2


In [12]:
_usecols = ['home', 'area_id']
merged_only[merged_only['date']==1][_usecols]

,home,area_id
0,1,1
1,1,1
2,2,1
3,1,2
4,2,2


In [13]:
merged_only[merged_only['date']==2][_usecols]

,home,area_id
5,1,1
6,1,1
7,2,1
8,1,2
9,2,2


In [14]:
merged_only[merged_only['date']==3][_usecols]

,home,area_id
10,1,1
11,1,3
12,3,3
13,1,3
14,2,3
